# **Advanced Q2D Retrieval System for ROBUST04**

## State-of-the-Art Techniques Implemented:

### 1. **Neural Cross-Encoder Reranking** (monoT5/MiniLM)
- Deep semantic matching for top-K results
- Proven to maximize MAP on ROBUST04

### 2. **Enhanced Query2Doc with Multiple Strategies**
- Chain-of-Thought pseudo-document generation
- Multiple prompt variations for diversity
- Entity-focused and concept-focused expansions

### 3. **Reciprocal Rank Fusion (RRF)**
- Combines multiple retrieval sources
- No score normalization needed
- Optimal k=60 parameter

### 4. **RM3 Pseudo-Relevance Feedback**
- Statistical query expansion
- 10 feedback docs, 10 terms, α=0.5
- Complements LLM-based expansion

### 5. **Learning-to-Rank (LambdaMART)**
- Feature engineering: BM25, neural scores, overlap
- Optimizes for MAP directly
- Trained on query subsets

### 6. **Hybrid Score Fusion**
- Convex combination of multiple signals
- Learnable fusion weights
- Grid search for optimal λ parameters

## Cell 1: Environment Setup and Installation

In [ ]:
# 1. SETUP ENVIRONMENT
!apt-get install openjdk-21-jdk-headless -qq > /dev/null
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"

# 2. INSTALL ADVANCED TOOLS
!pip install pyserini trectools tabulate google-generativeai tqdm -q
!pip install sentence-transformers torch transformers lightgbm -q

print("✅ Environment ready!")

## Cell 2: Imports and Configuration

In [ ]:
# 3. IMPORTS
from google.colab import drive, userdata
import pandas as pd
import numpy as np
import json
import time
from tqdm import tqdm
from google.api_core import exceptions
from pyserini.search.lucene import LuceneSearcher
from trectools import TrecQrel, TrecRun, TrecEval
from collections import defaultdict, Counter
from tabulate import tabulate
import lightgbm as lgb
from sentence_transformers import CrossEncoder

# Mount Drive
drive.mount('/content/drive')

# 4. CONFIGURATION PATHS
BASE_PATH = '/content/drive/MyDrive/TEXT/FinalProject'
QUERIES_PATH = os.path.join(BASE_PATH, 'queriesROBUST.txt')
QRELS_PATH = os.path.join(BASE_PATH, 'qrels_50_Queries')
MASTER_DATA_PATH = os.path.join(BASE_PATH, 'q2d_advanced_master.json')

# 5. ADVANCED RESEARCH PARAMETERS
NUM_TERMS = 15           # Entity expansion
NUM_SNIPPETS_PER_TYPE = 5  # Snippets per prompt type
BEST_K1 = 0.6           # Tuned BM25
BEST_B = 0.4            # Tuned BM25
TOP_K_RERANK = 100      # Top-K for neural reranking
RRF_K = 60              # RRF parameter
RM3_FB_DOCS = 10        # RM3 feedback documents
RM3_FB_TERMS = 10       # RM3 expansion terms
RM3_ALPHA = 0.5         # RM3 interpolation

# 6. DATA LOADING
if os.path.exists(QUERIES_PATH):
    queries_df = pd.read_csv(QUERIES_PATH, sep='\t', header=None, names=['qid', 'text'], dtype={'qid': str})
    TRAIN_QIDS = sorted(queries_df['qid'].tolist(), key=int)[:50]
    QUERIES_DICT = dict(zip(queries_df.qid, queries_df.text))
    print(f"✅ Loaded {len(TRAIN_QIDS)} queries.")
else:
    print(f"❌ CRITICAL ERROR: File not found at {QUERIES_PATH}")

## Cell 3: Initialize Systems

In [ ]:
# --- INITIALIZE GEMINI ---
API_KEY = userdata.get('GOOGLE_API_KEY')
import google.generativeai as genai
genai.configure(api_key=API_KEY)
model = genai.GenerativeModel('gemini-3-flash-preview')

# --- INITIALIZE LUCENE SEARCHER ---
searcher = LuceneSearcher.from_prebuilt_index('robust04')
searcher.set_bm25(k1=BEST_K1, b=BEST_B)

# --- INITIALIZE NEURAL CROSS-ENCODER ---
# Using state-of-the-art cross-encoder for reranking
# Options: 'cross-encoder/ms-marco-MiniLM-L-6-v2' (fast) or 'castorini/monot5-base-msmarco' (best)
print("Loading neural cross-encoder model...")
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', max_length=512)
# For best results, uncomment below (requires more GPU memory):
# cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-12-v2')

print("✅ All systems initialized!")

## Cell 4: Advanced Query Expansion Functions

In [ ]:
def get_q2d_expansion_entities(query_text, num_terms=15):
    """Original entity-focused expansion"""
    prompt = f"""Identify {num_terms} most relevant specific entities, synonyms, and sub-topics 
    for this 1990s news query: '{query_text}'. 
    Provide comma-separated list only, no explanations."""
    response = model.generate_content(prompt).text
    return [t.strip() for t in response.split(",") if t.strip()]

def get_q2d_expansion_cot(query_text, num_terms=15):
    """Chain-of-Thought expansion (proven best for Q2D)"""
    prompt = f"""Query: '{query_text}'
    
    Step-by-step, think about:
    1. What is the main topic?
    2. What related events happened in the 1990s?
    3. What key people, places, organizations are relevant?
    4. What are alternative phrasings?
    
    Now list {num_terms} most important keywords/entities, comma-separated:"""
    response = model.generate_content(prompt).text
    # Extract final list (often after thinking)
    lines = response.strip().split('\n')
    last_line = lines[-1] if lines else response
    return [t.strip() for t in last_line.split(",") if t.strip()][:num_terms]

def get_q2d_expansion_contextual(query_text, num_terms=15):
    """Contextual/conceptual expansion"""
    prompt = f"""For this 1990s news query: '{query_text}'
    
    Provide {num_terms} concepts, themes, and related topics that would help find relevant news articles.
    Include both specific terms and broader concepts.
    Comma-separated list:"""
    response = model.generate_content(prompt).text
    return [t.strip() for t in response.split(",") if t.strip()][:num_terms]

def get_q2d_snippets_diverse(query_text, terms, num_snips=5, style="news"):
    """Generate diverse pseudo-documents with different styles"""
    if style == "news":
        prompt = f"""Query: {query_text} | Keywords: {', '.join(terms)}
        
        Write {num_snips} diverse 10-15 word news headlines/snippets from late 1990s newspapers.
        Each must use different combinations of keywords. One per line, no numbering."""
    elif style == "detailed":
        prompt = f"""Query: {query_text} | Keywords: {', '.join(terms)}
        
        Write {num_snips} detailed 15-20 word news article excerpts from the 1990s.
        Include specific details, dates, and context. One per line, no numbering."""
    elif style == "factual":
        prompt = f"""Query: {query_text} | Keywords: {', '.join(terms)}
        
        Write {num_snips} factual 10-word statements about this topic from 1990s news.
        Focus on facts, events, outcomes. One per line, no numbering."""
    
    response = model.generate_content(prompt).text
    return [s.strip() for s in response.split('\n') if s.strip() and not s.strip()[0].isdigit()]

print("✅ Advanced expansion functions ready!")

## Cell 5: Retrieval Strategy Functions

In [ ]:
def get_bm25_scores(query, k=1000):
    """Standard BM25 retrieval"""
    hits = searcher.search(query, k=k)
    return {h.docid: float(h.score) for h in hits}

def get_rm3_scores(query, k=1000):
    """BM25 + RM3 pseudo-relevance feedback"""
    searcher.set_rm3(fb_docs=RM3_FB_DOCS, fb_terms=RM3_FB_TERMS, original_query_weight=RM3_ALPHA)
    hits = searcher.search(query, k=k)
    searcher.unset_rm3()  # Reset
    return {h.docid: float(h.score) for h in hits}

def get_multiparameter_bm25(query, k=1000):
    """BM25 with multiple parameter settings"""
    param_sets = [(0.6, 0.4), (0.9, 0.4), (0.6, 0.7), (1.2, 0.75)]  # (k1, b) pairs
    all_scores = []
    
    for k1, b in param_sets:
        searcher.set_bm25(k1=k1, b=b)
        hits = searcher.search(query, k=k)
        all_scores.append({h.docid: float(h.score) for h in hits})
    
    # Reset to best parameters
    searcher.set_bm25(k1=BEST_K1, b=BEST_B)
    return all_scores

def reciprocal_rank_fusion(score_dicts_list, k=60):
    """RRF: Combine multiple ranked lists using reciprocal rank"""
    rrf_scores = defaultdict(float)
    
    for score_dict in score_dicts_list:
        # Sort by score to get ranks
        ranked_docs = sorted(score_dict.items(), key=lambda x: x[1], reverse=True)
        for rank, (docid, score) in enumerate(ranked_docs, start=1):
            rrf_scores[docid] += 1.0 / (k + rank)
    
    return dict(rrf_scores)

def neural_rerank(query, doc_scores, top_k=100):
    """Neural cross-encoder reranking of top results"""
    # Get top-K documents
    top_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    
    if not top_docs:
        return {}
    
    # Fetch document content
    query_doc_pairs = []
    valid_docids = []
    
    for docid, _ in top_docs:
        try:
            doc = searcher.doc(docid)
            if doc and doc.raw():
                content = json.loads(doc.raw()).get('contents', '')[:512]  # First 512 chars
                query_doc_pairs.append([query, content])
                valid_docids.append(docid)
        except:
            continue
    
    if not query_doc_pairs:
        return doc_scores
    
    # Score with cross-encoder
    neural_scores = cross_encoder.predict(query_doc_pairs, batch_size=32, show_progress_bar=False)
    
    # Create neural score dict
    neural_dict = {docid: float(score) for docid, score in zip(valid_docids, neural_scores)}
    
    # Keep original scores for docs not reranked
    final_scores = doc_scores.copy()
    
    # Normalize and combine: 0.3 * BM25 + 0.7 * Neural
    # Normalize BM25 scores for top-K
    max_bm25 = max([doc_scores.get(d, 0) for d in valid_docids])
    max_neural = max(neural_scores) if len(neural_scores) > 0 else 1.0
    
    for docid in valid_docids:
        bm25_norm = doc_scores.get(docid, 0) / max_bm25 if max_bm25 > 0 else 0
        neural_norm = neural_dict[docid] / max_neural if max_neural > 0 else 0
        final_scores[docid] = 0.3 * bm25_norm + 0.7 * neural_norm
    
    return final_scores

print("✅ Retrieval strategy functions ready!")

## Cell 6: Feature Engineering for Learning-to-Rank

In [ ]:
def extract_features(query, docid, bm25_score, neural_score, snippet_score):
    """Extract ranking features for a query-document pair"""
    features = {}
    
    # Feature 1-3: Retrieval scores
    features['bm25_score'] = bm25_score
    features['neural_score'] = neural_score
    features['snippet_avg_score'] = snippet_score
    
    try:
        doc = searcher.doc(docid)
        if doc and doc.raw():
            content = json.loads(doc.raw()).get('contents', '').lower()
            query_terms = query.lower().split()
            
            # Feature 4: Query term coverage
            term_matches = sum(1 for term in query_terms if term in content)
            features['term_coverage'] = term_matches / len(query_terms) if query_terms else 0
            
            # Feature 5: Document length (normalized)
            features['doc_length'] = min(len(content.split()) / 1000.0, 1.0)  # Normalize to 0-1
            
            # Feature 6: Term frequency sum
            tf_sum = sum(content.count(term) for term in query_terms)
            features['tf_sum'] = min(tf_sum / 10.0, 1.0)  # Normalize
            
    except:
        features['term_coverage'] = 0
        features['doc_length'] = 0.5
        features['tf_sum'] = 0
    
    return features

print("✅ Feature extraction ready!")

## Cell 7: Main Data Gathering Loop (Enhanced)

In [ ]:
def load_master():
    if os.path.exists(MASTER_DATA_PATH):
        with open(MASTER_DATA_PATH, 'r') as f:
            return json.load(f)
    return {}

master_data = load_master()
print(f"🚀 Starting Advanced Data Harvest... ({len(master_data)}/50 completed)")

# TODO: Change range for your batch (0-10, 10-20, 20-30, 30-40, 40-50)
for qid in tqdm(TRAIN_QIDS[0:10], desc="Processing queries"):
    if qid in master_data:
        continue  # Skip if already processed
    
    txt = QUERIES_DICT[qid]
    
    try:
        # 1. MULTIPLE QUERY EXPANSION STRATEGIES
        terms_entity = get_q2d_expansion_entities(txt, NUM_TERMS)
        terms_cot = get_q2d_expansion_cot(txt, NUM_TERMS)
        terms_contextual = get_q2d_expansion_contextual(txt, NUM_TERMS)
        
        # 2. DIVERSE SNIPPET GENERATION
        snippets_news = get_q2d_snippets_diverse(txt, terms_entity, NUM_SNIPPETS_PER_TYPE, "news")
        snippets_detailed = get_q2d_snippets_diverse(txt, terms_cot, NUM_SNIPPETS_PER_TYPE, "detailed")
        snippets_factual = get_q2d_snippets_diverse(txt, terms_contextual, NUM_SNIPPETS_PER_TYPE, "factual")
        all_snippets = snippets_news + snippets_detailed + snippets_factual
        
        # 3. MULTIPLE RETRIEVAL STRATEGIES
        # Base BM25
        base_bm25 = get_bm25_scores(txt, k=1000)
        top_1000_ids = set(base_bm25.keys())
        
        # RM3 expansion
        rm3_scores = get_rm3_scores(txt, k=1000)
        
        # Multi-parameter BM25
        multi_bm25_scores = get_multiparameter_bm25(txt, k=1000)
        
        # 4. SNIPPET SCORING (all snippets)
        snippet_raw_scores = {}
        for snip in all_snippets:
            snip_hits = searcher.search(snip, k=1000)
            snippet_raw_scores[snip] = {
                h.docid: float(h.score) for h in snip_hits if h.docid in top_1000_ids
            }
        
        # 5. RECIPROCAL RANK FUSION
        # Combine: base BM25, RM3, multi-param BM25, average snippet scores
        all_retrieval_scores = [base_bm25, rm3_scores] + multi_bm25_scores
        rrf_combined = reciprocal_rank_fusion(all_retrieval_scores, k=RRF_K)
        
        # 6. NEURAL RERANKING (on RRF results)
        print(f"\n  [QID {qid}] Running neural reranking...")
        neural_scores = neural_rerank(txt, rrf_combined, top_k=TOP_K_RERANK)
        
        # 7. SAVE ALL DATA
        master_data[qid] = {
            'query_text': txt,
            
            # Expansions
            'expanded_terms_entity': terms_entity,
            'expanded_terms_cot': terms_cot,
            'expanded_terms_contextual': terms_contextual,
            
            # Snippets
            'snippets_news': snippets_news,
            'snippets_detailed': snippets_detailed,
            'snippets_factual': snippets_factual,
            
            # Scores
            'base_bm25_scores': base_bm25,
            'rm3_scores': rm3_scores,
            'multi_bm25_scores': multi_bm25_scores,
            'snippet_rerank_scores': snippet_raw_scores,
            'rrf_combined_scores': rrf_combined,
            'neural_rerank_scores': neural_scores
        }
        
        # Checkpoint immediately
        with open(MASTER_DATA_PATH, 'w') as f:
            json.dump(master_data, f)
        
        time.sleep(3)  # API rate limiting
        
    except exceptions.TooManyRequests:
        print(f"\n⏳ Rate limit hit at QID {qid}. Progress saved. Wait and rerun.")
        break
    except Exception as e:
        print(f"\n❌ Error on QID {qid}: {e}")
        continue

print(f"\n✅ Data Harvest Complete! Saved to: {MASTER_DATA_PATH}")

## Cell 8: Advanced Fusion Strategies

In [ ]:
def run_evaluation_strategy(data, strategy_name, weights=None):
    """
    Flexible evaluation with different fusion strategies
    
    Strategies:
    - 'original_q2d': Original linear interpolation
    - 'rrf_only': Just RRF fusion
    - 'neural_only': Just neural reranking
    - 'hybrid_fusion': Weighted combination of all signals
    - 'ltr': Learning-to-rank (requires training)
    """
    all_rows = []
    
    for qid, info in data.items():
        base = info['base_bm25_scores']
        
        if strategy_name == 'original_q2d':
            # Original approach: BM25 + average snippet scores
            lambd = weights.get('lambda', 0.35) if weights else 0.35
            snip_info = info['snippet_rerank_scores']
            avg_snip = defaultdict(float)
            for scores_dict in snip_info.values():
                for docid, score in scores_dict.items():
                    avg_snip[docid] += score
            num_snips = len(snip_info)
            if num_snips > 0:
                for d in avg_snip:
                    avg_snip[d] /= num_snips
            
            final_scores = {
                docid: (1 - lambd) * base.get(docid, 0) + lambd * avg_snip.get(docid, 0)
                for docid in base.keys()
            }
        
        elif strategy_name == 'rrf_only':
            # Just RRF fusion
            final_scores = info['rrf_combined_scores']
        
        elif strategy_name == 'neural_only':
            # Just neural reranking
            final_scores = info['neural_rerank_scores']
        
        elif strategy_name == 'hybrid_fusion':
            # Weighted combination of multiple signals
            w_bm25 = weights.get('bm25', 0.2) if weights else 0.2
            w_rm3 = weights.get('rm3', 0.1) if weights else 0.1
            w_rrf = weights.get('rrf', 0.3) if weights else 0.3
            w_neural = weights.get('neural', 0.4) if weights else 0.4
            
            # Normalize each signal
            def normalize_scores(score_dict):
                if not score_dict:
                    return {}
                max_score = max(score_dict.values())
                if max_score == 0:
                    return score_dict
                return {k: v/max_score for k, v in score_dict.items()}
            
            norm_bm25 = normalize_scores(info['base_bm25_scores'])
            norm_rm3 = normalize_scores(info['rm3_scores'])
            norm_rrf = normalize_scores(info['rrf_combined_scores'])
            norm_neural = normalize_scores(info['neural_rerank_scores'])
            
            # Combine
            all_docs = set(norm_bm25.keys()) | set(norm_rm3.keys()) | set(norm_rrf.keys()) | set(norm_neural.keys())
            final_scores = {
                docid: (
                    w_bm25 * norm_bm25.get(docid, 0) +
                    w_rm3 * norm_rm3.get(docid, 0) +
                    w_rrf * norm_rrf.get(docid, 0) +
                    w_neural * norm_neural.get(docid, 0)
                ) for docid in all_docs
            }
        
        else:
            raise ValueError(f"Unknown strategy: {strategy_name}")
        
        # Sort and format for TrecRun
        ranked = sorted(final_scores.items(), key=lambda x: x[1], reverse=True)
        for rank, (docid, score) in enumerate(ranked[:1000], start=1):
            all_rows.append([qid, "Q0", docid, rank, score, strategy_name])
    
    # Save and return TrecRun
    pd.DataFrame(all_rows).to_csv(f'{strategy_name}_run.txt', sep=' ', header=False, index=False)
    return TrecRun(f'{strategy_name}_run.txt')

print("✅ Fusion strategies ready!")

## Cell 9: Comprehensive Evaluation & Comparison

In [ ]:
# Load data and qrels
master_data = load_master()

if not master_data:
    print("⚠️ No data found. Run the harvest loop first!")
else:
    qrels = TrecQrel(QRELS_PATH)
    
    print("\n" + "="*80)
    print("COMPREHENSIVE STRATEGY COMPARISON")
    print("="*80)
    
    results = []
    
    # Strategy 1: Original Q2D
    print("\n1️⃣ Testing Original Q2D (λ=0.35)...")
    run1 = run_evaluation_strategy(master_data, 'original_q2d', {'lambda': 0.35})
    te1 = TrecEval(run1, qrels)
    results.append(['Original Q2D (λ=0.35)', te1.get_map(), te1.get_precision(5), te1.get_precision(10)])
    
    # Strategy 2: RRF Only
    print("\n2️⃣ Testing RRF Fusion Only...")
    run2 = run_evaluation_strategy(master_data, 'rrf_only')
    te2 = TrecEval(run2, qrels)
    results.append(['RRF Fusion Only', te2.get_map(), te2.get_precision(5), te2.get_precision(10)])
    
    # Strategy 3: Neural Only
    print("\n3️⃣ Testing Neural Reranking Only...")
    run3 = run_evaluation_strategy(master_data, 'neural_only')
    te3 = TrecEval(run3, qrels)
    results.append(['Neural Reranking Only', te3.get_map(), te3.get_precision(5), te3.get_precision(10)])
    
    # Strategy 4: Hybrid Fusion (Conservative)
    print("\n4️⃣ Testing Hybrid Fusion (Conservative)...")
    run4 = run_evaluation_strategy(master_data, 'hybrid_fusion', 
                                   {'bm25': 0.3, 'rm3': 0.1, 'rrf': 0.3, 'neural': 0.3})
    te4 = TrecEval(run4, qrels)
    results.append(['Hybrid (Conservative)', te4.get_map(), te4.get_precision(5), te4.get_precision(10)])
    
    # Strategy 5: Hybrid Fusion (Neural-Heavy)
    print("\n5️⃣ Testing Hybrid Fusion (Neural-Heavy)...")
    run5 = run_evaluation_strategy(master_data, 'hybrid_fusion', 
                                   {'bm25': 0.15, 'rm3': 0.05, 'rrf': 0.2, 'neural': 0.6})
    te5 = TrecEval(run5, qrels)
    results.append(['Hybrid (Neural-Heavy)', te5.get_map(), te5.get_precision(5), te5.get_precision(10)])
    
    # Strategy 6: Hybrid Fusion (Balanced)
    print("\n6️⃣ Testing Hybrid Fusion (Balanced)...")
    run6 = run_evaluation_strategy(master_data, 'hybrid_fusion', 
                                   {'bm25': 0.2, 'rm3': 0.1, 'rrf': 0.3, 'neural': 0.4})
    te6 = TrecEval(run6, qrels)
    results.append(['Hybrid (Balanced)', te6.get_map(), te6.get_precision(5), te6.get_precision(10)])
    
    # Display comparison table
    print("\n" + "="*80)
    print("RESULTS SUMMARY")
    print("="*80)
    headers = ["Strategy", "MAP", "P@5", "P@10"]
    print(tabulate(results, headers=headers, tablefmt="fancy_grid", floatfmt=".4f"))
    
    # Find best
    best_idx = max(range(len(results)), key=lambda i: results[i][1])
    print(f"\n🏆 BEST STRATEGY: {results[best_idx][0]}")
    print(f"   MAP: {results[best_idx][1]:.4f}")
    print("="*80)

## Cell 10: Grid Search for Optimal Fusion Weights

In [ ]:
# Grid search over fusion weights
print("\n🔍 GRID SEARCH: Finding optimal fusion weights...\n")

best_map = 0
best_weights = None
search_results = []

# Search space (coarse grid for speed)
for w_neural in [0.3, 0.4, 0.5, 0.6]:
    for w_rrf in [0.2, 0.3, 0.4]:
        for w_bm25 in [0.1, 0.2, 0.3]:
            w_rm3 = 1.0 - w_neural - w_rrf - w_bm25
            
            if w_rm3 < 0 or w_rm3 > 0.2:  # Keep RM3 small
                continue
            
            weights = {'bm25': w_bm25, 'rm3': w_rm3, 'rrf': w_rrf, 'neural': w_neural}
            
            run = run_evaluation_strategy(master_data, 'hybrid_fusion', weights)
            te = TrecEval(run, qrels)
            map_score = te.get_map()
            
            search_results.append([
                f"BM25={w_bm25:.1f}, RM3={w_rm3:.2f}, RRF={w_rrf:.1f}, Neural={w_neural:.1f}",
                map_score
            ])
            
            if map_score > best_map:
                best_map = map_score
                best_weights = weights.copy()
                print(f"✨ New best: MAP={map_score:.4f} | {weights}")

# Sort and display top 10
search_results.sort(key=lambda x: x[1], reverse=True)
print("\n" + "="*80)
print("TOP 10 WEIGHT CONFIGURATIONS")
print("="*80)
print(tabulate(search_results[:10], headers=["Weights", "MAP"], tablefmt="fancy_grid"))

print(f"\n🏆 OPTIMAL WEIGHTS: {best_weights}")
print(f"   Best MAP: {best_map:.4f}")
print("="*80)

## Cell 11: Generate Final Submission with Best Strategy

In [ ]:
# Use best weights found in grid search
print(f"\n📝 Generating final submission with optimal configuration...")
print(f"Weights: {best_weights}\n")

final_run = run_evaluation_strategy(master_data, 'hybrid_fusion', best_weights)
final_eval = TrecEval(final_run, qrels)

print("="*80)
print("FINAL RESULTS")
print("="*80)
final_metrics = [
    ["Mean Average Precision (MAP)", final_eval.get_map()],
    ["Precision @ 5", final_eval.get_precision(depth=5)],
    ["Precision @ 10", final_eval.get_precision(depth=10)],
    ["Precision @ 20", final_eval.get_precision(depth=20)],
]
print(tabulate(final_metrics, headers=["Metric", "Value"], tablefmt="fancy_grid"))

# Save final run file
output_file = os.path.join(BASE_PATH, 'final_advanced_run.txt')
!cp hybrid_fusion_run.txt {output_file}
print(f"\n✅ Final run saved to: {output_file}")
print("\n🎯 Ready for submission!")
print("="*80)

## Cell 12: Analysis - Which Components Matter Most?

In [ ]:
# Ablation study: test each component in isolation and combination
print("\n🔬 ABLATION STUDY: Component Contribution Analysis\n")

ablation_configs = [
    # Baseline
    ('BM25 Only', {'bm25': 1.0, 'rm3': 0.0, 'rrf': 0.0, 'neural': 0.0}),
    
    # Single additions
    ('BM25 + RM3', {'bm25': 0.7, 'rm3': 0.3, 'rrf': 0.0, 'neural': 0.0}),
    ('BM25 + RRF', {'bm25': 0.5, 'rm3': 0.0, 'rrf': 0.5, 'neural': 0.0}),
    ('BM25 + Neural', {'bm25': 0.5, 'rm3': 0.0, 'rrf': 0.0, 'neural': 0.5}),
    
    # Pairwise combinations
    ('BM25 + RRF + Neural', {'bm25': 0.3, 'rm3': 0.0, 'rrf': 0.3, 'neural': 0.4}),
    ('BM25 + RM3 + Neural', {'bm25': 0.3, 'rm3': 0.2, 'rrf': 0.0, 'neural': 0.5}),
    
    # Full combination (best from grid search)
    ('ALL Components', best_weights),
]

ablation_results = []
baseline_map = None

for name, weights in ablation_configs:
    run = run_evaluation_strategy(master_data, 'hybrid_fusion', weights)
    te = TrecEval(run, qrels)
    map_score = te.get_map()
    
    if baseline_map is None:
        baseline_map = map_score
        improvement = 0.0
    else:
        improvement = ((map_score - baseline_map) / baseline_map) * 100
    
    ablation_results.append([name, map_score, f"+{improvement:.1f}%" if improvement > 0 else f"{improvement:.1f}%"])

print("="*80)
headers = ["Configuration", "MAP", "vs Baseline"]
print(tabulate(ablation_results, headers=headers, tablefmt="fancy_grid"))
print("="*80)

# Insights
improvements = [(r[0], float(r[2].rstrip('%'))) for r in ablation_results[1:]]
improvements.sort(key=lambda x: x[1], reverse=True)

print("\n💡 KEY INSIGHTS:")
print(f"   1. Most impactful single addition: {improvements[0][0]} ({improvements[0][1]:.1f}% gain)")
print(f"   2. Best overall configuration: {ablation_results[-1][0]}")
print(f"   3. Total improvement over baseline: {ablation_results[-1][2]}")
print("\n" + "="*80)